In [1]:
import pandas as pd
import numpy as np

print("Target engineering started.")

Target engineering started.


In [2]:
assets = pd.read_csv(
    "../data/processed/assets_clean.csv"
)

assets["last_maintenance_date"] = pd.to_datetime(
    assets["last_maintenance_date"]
)

print(assets)

  asset_id   section_id   department asset_type  installation_year  \
0   TRK001   NDL-MTJ-01  ENGINEERING      TRACK               2012   
1   TRK002   NDL-MTJ-02  ENGINEERING      TRACK               2020   
2   SIG001   MTJ-AGC-01          S&T     SIGNAL               2016   
3   SIG002   AGC-GWL-01          S&T     SIGNAL               2022   
4   OHE001   GWL-JHS-01     TRACTION        OHE               2014   
5   OHE002  JHS-BINA-01     TRACTION        OHE               2019   

   criticality  condition_score  failure_count last_maintenance_date  
0           10               62              7            2026-05-12  
1            8               88              1            2026-07-10  
2            9               71              4            2026-06-15  
3            7               94              0            2026-08-01  
4            9               68              5            2026-05-20  
5            8               91              1            2026-07-22  


In [3]:
snapshot_dates = pd.date_range(
    start="2020-01-01",
    end="2026-06-01",
    freq="MS"
)

print(snapshot_dates[:10])
print("Number of snapshots:", len(snapshot_dates))

DatetimeIndex(['2020-01-01', '2020-02-01', '2020-03-01', '2020-04-01',
               '2020-05-01', '2020-06-01', '2020-07-01', '2020-08-01',
               '2020-09-01', '2020-10-01'],
              dtype='datetime64[us]', freq='MS')
Number of snapshots: 78


In [4]:
asset_snapshots = assets[
    [
        "asset_id",
        "section_id",
        "department",
        "asset_type",
        "installation_year",
        "criticality"
    ]
].copy()

asset_snapshots["key"] = 1

dates = pd.DataFrame({
    "snapshot_date": snapshot_dates
})

dates["key"] = 1

historical = asset_snapshots.merge(
    dates,
    on="key"
)

historical.drop(
    columns=["key"],
    inplace=True
)

print(historical.head())
print("Rows:", len(historical))

  asset_id  section_id   department asset_type  installation_year  \
0   TRK001  NDL-MTJ-01  ENGINEERING      TRACK               2012   
1   TRK001  NDL-MTJ-01  ENGINEERING      TRACK               2012   
2   TRK001  NDL-MTJ-01  ENGINEERING      TRACK               2012   
3   TRK001  NDL-MTJ-01  ENGINEERING      TRACK               2012   
4   TRK001  NDL-MTJ-01  ENGINEERING      TRACK               2012   

   criticality snapshot_date  
0           10    2020-01-01  
1           10    2020-02-01  
2           10    2020-03-01  
3           10    2020-04-01  
4           10    2020-05-01  
Rows: 468


In [5]:
historical["asset_age_years"] = (
    historical["snapshot_date"].dt.year
    - historical["installation_year"]
)

historical["asset_age_years"] = (
    historical["asset_age_years"].clip(lower=0)
)

print(
    historical[
        [
            "asset_id",
            "snapshot_date",
            "asset_age_years"
        ]
    ].head(20)
)

   asset_id snapshot_date  asset_age_years
0    TRK001    2020-01-01                8
1    TRK001    2020-02-01                8
2    TRK001    2020-03-01                8
3    TRK001    2020-04-01                8
4    TRK001    2020-05-01                8
5    TRK001    2020-06-01                8
6    TRK001    2020-07-01                8
7    TRK001    2020-08-01                8
8    TRK001    2020-09-01                8
9    TRK001    2020-10-01                8
10   TRK001    2020-11-01                8
11   TRK001    2020-12-01                8
12   TRK001    2021-01-01                9
13   TRK001    2021-02-01                9
14   TRK001    2021-03-01                9
15   TRK001    2021-04-01                9
16   TRK001    2021-05-01                9
17   TRK001    2021-06-01                9
18   TRK001    2021-07-01                9
19   TRK001    2021-08-01                9


In [6]:
failures = pd.read_csv(
    "../data/processed/failures_clean.csv"
)

failures["failure_date"] = pd.to_datetime(
    failures["failure_date"]
)

In [7]:
historical["historical_failure_count"] = 0

for idx, row in historical.iterrows():

    count = failures[
        (failures["asset_id"] == row["asset_id"]) &
        (failures["failure_date"] < row["snapshot_date"])
    ].shape[0]

    historical.loc[
        idx,
        "historical_failure_count"
    ] = count

In [8]:
historical["failure_within_30_days"] = 0

for idx, row in historical.iterrows():

    future_failure = failures[
        (failures["asset_id"] == row["asset_id"]) &
        (failures["failure_date"] >= row["snapshot_date"]) &
        (
            failures["failure_date"]
            <= row["snapshot_date"] +
            pd.Timedelta(days=30)
        )
    ]

    if len(future_failure) > 0:
        historical.loc[
            idx,
            "failure_within_30_days"
        ] = 1

In [9]:
print(
    historical[
        [
            "asset_id",
            "snapshot_date",
            "historical_failure_count",
            "failure_within_30_days"
        ]
    ].head(30)
)

   asset_id snapshot_date  historical_failure_count  failure_within_30_days
0    TRK001    2020-01-01                         0                       0
1    TRK001    2020-02-01                         0                       0
2    TRK001    2020-03-01                         0                       0
3    TRK001    2020-04-01                         0                       0
4    TRK001    2020-05-01                         0                       0
5    TRK001    2020-06-01                         0                       0
6    TRK001    2020-07-01                         0                       0
7    TRK001    2020-08-01                         0                       0
8    TRK001    2020-09-01                         0                       0
9    TRK001    2020-10-01                         0                       0
10   TRK001    2020-11-01                         0                       0
11   TRK001    2020-12-01                         0                       0
12   TRK001 

In [10]:
print(
    historical["failure_within_30_days"]
    .value_counts()
)

print(
    historical["failure_within_30_days"]
    .value_counts(normalize=True)
)

failure_within_30_days
0    461
1      7
Name: count, dtype: int64
failure_within_30_days
0    0.985043
1    0.014957
Name: proportion, dtype: float64


In [11]:
historical = historical[
    historical["snapshot_date"].dt.year
    >= historical["installation_year"]
].copy()

In [12]:
print(
    historical[
        [
            "asset_id",
            "installation_year",
            "snapshot_date"
        ]
    ].head(20)
)

   asset_id  installation_year snapshot_date
0    TRK001               2012    2020-01-01
1    TRK001               2012    2020-02-01
2    TRK001               2012    2020-03-01
3    TRK001               2012    2020-04-01
4    TRK001               2012    2020-05-01
5    TRK001               2012    2020-06-01
6    TRK001               2012    2020-07-01
7    TRK001               2012    2020-08-01
8    TRK001               2012    2020-09-01
9    TRK001               2012    2020-10-01
10   TRK001               2012    2020-11-01
11   TRK001               2012    2020-12-01
12   TRK001               2012    2021-01-01
13   TRK001               2012    2021-02-01
14   TRK001               2012    2021-03-01
15   TRK001               2012    2021-04-01
16   TRK001               2012    2021-05-01
17   TRK001               2012    2021-06-01
18   TRK001               2012    2021-07-01
19   TRK001               2012    2021-08-01


In [13]:
historical.to_csv(
    "../data/processed/failure_training_base.csv",
    index=False
)

print("✅ Failure training base saved.")

✅ Failure training base saved.
